# Midterm Practice Notebook: ARTI 310 - Programming for AI

Welcome to your midterm practice! This notebook links the concepts from Lectures 1-8 into a single, cohesive workflow. You will move from raw data to building a custom PyTorch dataset.

**How to use this notebook:**
1. Write your solution in the designated code blocks.
2. Run the cell.
3. If the `assert` statements pass without throwing an error, your code is correct!

## Task 1: Python Essentials & Data Structures

Let's start by simulating a raw data stream. Create a list of dictionaries where each dictionary represents a student. Each student needs the following keys:
- `'id'`: integer (e.g., 101, 102)
- `'grades'`: a list of three floats (e.g., `[85.5, 90.0, 78.5]`)
- `'passed'`: a boolean indicating if their average grade is >= 60.0
- `'major'`: a string (e.g., `'AI'`, `'CYS'`)

Create a list named `raw_students` containing exactly 3 student dictionaries.

In [1]:
from Lab8.lab08 import labels

# TODO: Create your raw_students list of dictionaries here
raw_students = [
    {"id":"1", "grades":[85.5, 90, 78.5], "passed":True, "major":"AI"},
    {"id":"2", "grades":[40.5, 40, 50.5], "passed":False, "major":"CYS"},
    {"id":"4", "grades":[95.5, 90, 80.5], "passed":True, "major":"CYS"}
]


# --- ASSERTION BLOCK ---
assert len(raw_students) == 3, "List must contain exactly 3 students."
assert all(isinstance(s, dict) for s in raw_students), "Each item must be a dictionary."
assert all(k in raw_students[0] for k in ['id', 'grades', 'passed', 'major']), "Missing required keys."
assert isinstance(raw_students[0]['grades'], list), "'grades' must be a list."
assert len(raw_students[0]['grades']) == 3, "Each student must have exactly 3 grades."
print("Task 1: Passed! ✓")

Task 1: Passed! ✓


## Task 2: NumPy & Scientific Computing

Extract all the grades from `raw_students` and convert them into a 2D NumPy array named `grades_matrix`. 

Then, use a built-in NumPy function to calculate the **mean grade for each student** (average across the rows), and store the result in a 1D NumPy array named `student_averages`.

In [2]:
import numpy as np

# TODO: Create grades_matrix and calculate student_averages
grades_matrix = np.array([row['grades'] for row in raw_students])
student_averages = grades_matrix.mean(axis=1)



# --- ASSERTION BLOCK ---
assert 'grades_matrix' in locals(), "grades_matrix not defined."
assert 'student_averages' in locals(), "student_averages not defined."
assert isinstance(grades_matrix, np.ndarray), "grades_matrix must be a NumPy array."
assert grades_matrix.shape == (3, 3), "grades_matrix should have shape (3, 3)."
assert student_averages.shape == (3,), "student_averages should have shape (3,)."
assert np.allclose(student_averages, np.mean(grades_matrix, axis=1)), "Averages calculated incorrectly."
print("Task 2: Passed! ✓")

Task 2: Passed! ✓


## Task 3: Data Engineering with Pandas

Convert your `raw_students` list into a Pandas DataFrame named `df`.

Next, perform the following data cleaning/engineering steps:
1. Extract the first grade from the 'grades' list into a new column called `Grade_1`.
2. Extract the second grade into a new column called `Grade_2`.
3. Extract the third grade into a new column called `Grade_3`.
4. Drop the original 'grades' column entirely.

In [8]:
import pandas as pd

# TODO: Create the DataFrame and perform the requested feature engineering
df = pd.DataFrame(raw_students, columns=['id', 'grades', 'passed', 'major'])
df["Grade_1"] = grades_matrix[:, 0]
df["Grade_2"] = grades_matrix[:, 1]
df["Grade_3"] = grades_matrix[:, 2]
df = df.drop("grades", axis=1)

print(df)

# --- ASSERTION BLOCK ---
assert 'df' in locals(), "DataFrame 'df' not defined."
assert isinstance(df, pd.DataFrame), "'df' must be a Pandas DataFrame."
assert 'grades' not in df.columns, "The original 'grades' column must be dropped."
assert all(col in df.columns for col in ['id', 'passed', 'major', 'Grade_1', 'Grade_2', 'Grade_3']), "Missing extracted grade columns."
assert df.shape == (3, 6), "DataFrame shape should be (3, 6)."
print("Task 3: Passed! ✓")

  id  passed major  Grade_1  Grade_2  Grade_3
0  1    True    AI     85.5     90.0     78.5
1  2   False   CYS     40.5     40.0     50.5
2  4    True   CYS     95.5     90.0     80.5
Task 3: Passed! ✓


## Task 4: Scikit-learn Preprocessing Pipeline

We need to prepare this DataFrame for a machine learning model. 
Using `ColumnTransformer`, create a preprocessing step named `preprocessor` that does the following:
1. Applies `StandardScaler` to the numerical columns: `['Grade_1', 'Grade_2', 'Grade_3']`.
2. Applies `OneHotEncoder` (with `sparse_output=False` or `sparse=False` depending on sklearn version) to the categorical column: `['major']`.

Finally, fit and transform your DataFrame (excluding the 'id' and 'passed' columns) using the preprocessor, saving the output array to `X_transformed`.

In [10]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# TODO: Build the ColumnTransformer and fit_transform the data
preprocessor = ColumnTransformer([
    ("scaler", StandardScaler(), ["Grade_1", "Grade_2", "Grade_3"]),
    ("encoder", OneHotEncoder(), ["major"])
])
X_transformed = preprocessor.fit_transform(df.drop(columns=['id', 'passed']))



# --- ASSERTION BLOCK ---
assert 'preprocessor' in locals(), "preprocessor not defined."
assert 'X_transformed' in locals(), "X_transformed not defined."
assert isinstance(preprocessor, ColumnTransformer), "Must be a ColumnTransformer object."
assert isinstance(X_transformed, np.ndarray), "Output must be a NumPy array."
assert X_transformed.shape[0] == 3, "Transformed array should have 3 rows."
assert X_transformed.shape[1] > 3, "Transformed array should have more than 3 columns (3 numerical + one-hot encoded columns)."
print("Task 4: Passed! ✓")

Task 4: Passed! ✓


## Task 5: PyTorch Custom Dataset

Now that our data is fully preprocessed into a numerical matrix (`X_transformed`), we need to load it into PyTorch.

Build a custom PyTorch Dataset class named `StudentDataset`.
It must inherit from `Dataset` and implement the three required methods: `__init__`, `__len__`, and `__getitem__`.

The target labels (`y`) should come from the `df['passed']` column (converted to integers 0 and 1).

Finally, instantiate `StudentDataset` using `X_transformed` and the labels, assigning it to a variable named `pytorch_dataset`.

In [16]:
import torch
from torch.utils.data import Dataset

# TODO: Create the StudentDataset class and instantiate it
class StudentDataset(Dataset):
    def __init__(self, features, labels):
        self.features = features
        self.labels = labels
    def __len__(self):
        return len(self.features)
    def __getitem__(self, idx):
        return self.features[idx], self.labels[idx]

features = torch.tensor(X_transformed, dtype=torch.float32)
labels = torch.tensor(df["passed"], dtype=torch.int64)
pytorch_dataset = StudentDataset(features, labels)



# --- ASSERTION BLOCK ---
assert 'StudentDataset' in locals(), "StudentDataset class not defined."
assert 'pytorch_dataset' in locals(), "pytorch_dataset instance not defined."
assert issubclass(StudentDataset, Dataset), "Class must inherit from torch.utils.data.Dataset."
assert len(pytorch_dataset) == 3, "Dataset length should be 3."

features, label = pytorch_dataset[0]
assert isinstance(features, torch.Tensor), "Features must be a PyTorch Tensor."
assert isinstance(label, torch.Tensor), "Label must be a PyTorch Tensor."
assert features.dtype == torch.float32 or features.dtype == torch.float64, "Features should be floating point tensors."
assert label.dtype == torch.int64 or label.dtype == torch.int32, "Labels should be integer tensors."
print("Task 5: Passed! ✓ You have successfully connected the entire AI data pipeline!")

Task 5: Passed! ✓ You have successfully connected the entire AI data pipeline!
